# Disease Scout PlantDoc Tomato Fine-tune

Runtime target: Google Colab T4 GPU. This notebook self-bootstraps from the pushed GitHub branch and the public PlantDoc dataset zip, then trains a small MobileNetV2 transfer-learning classifier for Disease Scout tomato disease classes.

In [ ]:
!nvidia-smi
!python --version

Clone the prototype branch that contains the training script. If this cell fails, the branch has not been pushed or Colab cannot reach GitHub.

In [ ]:
!rm -rf /content/t0mat0z
!git clone --depth 1 --branch colab-t4-finetune https://github.com/ShashwatM3/t0mat0z.git /content/t0mat0z
!ls -la /content/t0mat0z/training/disease-finetune

Download PlantDoc directly from the public dataset repository. This avoids local file upload and keeps the run reproducible.

In [ ]:
!rm -rf /content/PlantDoc-Dataset-master /content/PlantDoc-Dataset-master.zip
!wget -q --show-progress -O /content/PlantDoc-Dataset-master.zip https://github.com/pratikkayal/PlantDoc-Dataset/archive/refs/heads/master.zip
!python - <<'PY'
import zipfile
from pathlib import Path
zip_path = Path('/content/PlantDoc-Dataset-master.zip')
assert zip_path.exists() and zip_path.stat().st_size > 100_000_000, f'bad PlantDoc zip: {zip_path}'
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall('/content')
root = Path('/content/PlantDoc-Dataset-master')
assert (root / 'train').exists(), 'missing train folder after extract'
assert (root / 'test').exists(), 'missing test folder after extract'
print('PlantDoc extracted:', root)
PY

Run the small transfer-learning job. The defaults are intentionally short so the hackathon gets a real model artifact quickly.

In [ ]:
!python /content/t0mat0z/training/disease-finetune/train_colab_t4_plantdoc_tomato.py \
  --dataset-root /content/PlantDoc-Dataset-master \
  --output-dir /content/disease_scout_finetune \
  --image-size 224 \
  --batch-size 16 \
  --head-epochs 3 \
  --finetune-epochs 2 \
  --export-tflite

Inspect and download outputs. Keep `metrics.json`, `label_map.json`, and the `.keras` model as the run receipt.

In [ ]:
import json, pathlib
from google.colab import files
out = pathlib.Path('/content/disease_scout_finetune')
print((out / 'label_map.json').read_text())
print(json.dumps(json.loads((out / 'metrics.json').read_text()), indent=2)[:2000])
for name in ['disease_scout_tomato_mobilenetv2.keras', 'label_map.json', 'metrics.json', 'disease_scout_tomato_mobilenetv2.tflite']:
    path = out / name
    if path.exists():
        files.download(str(path))